In [15]:
from openai import OpenAI
import pandas as pd
import json
import time
import os
import re
import glob
from tqdm.auto import tqdm
from dotenv import load_dotenv
from datetime import datetime

# =========================
# 1) SETUP & CREDENTIALS
# =========================
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
if not API_KEY:
    raise ValueError("API Key not found! Check your .env file (OPENAI_API_KEY).")

client = OpenAI(api_key=API_KEY)

# =========================
# 2) CONFIG
# =========================
INPUT_DIR = r"D:\PythonEnv\Interpretable\hs_sample_400\GPTResult_MIXED_NOISE"

# 初始 batch（大也没关系，失败会自动拆分）
BATCH_SIZE = 50

# ✅ Change this anytime
MODEL_NAME = "gpt-5.2"   # e.g. "gpt-5.2", "gpt-3.5-turbo", etc.

# 输出上限（100条预测用不到 8000；2000 通常够）
MAX_OUTPUT_TOKENS = 2000

# 请求间隔（可调小/调大）
SLEEP_BETWEEN_CALLS = 0.2

# 失败日志目录（会保存失败批次和原始输出，方便 debug）
SAVE_FAIL_LOG = True

# =========================
# 3) Derived names
# =========================
def model_to_tag(model_name: str) -> str:
    tag = model_name.lower()
    tag = re.sub(r"[^a-z0-9]+", "_", tag)
    tag = re.sub(r"_+", "_", tag).strip("_")
    return tag

MODEL_TAG = model_to_tag(MODEL_NAME)
OUTPUT_DIR_NAME = f"GPTres_{MODEL_TAG}"            # e.g., GPTres_gpt3_5
OUTPUT_SUFFIX   = f"_{MODEL_TAG}"                 # e.g., _gpt3_5
OUTPUT_COL      = f"hatespeech_LLM_{MODEL_TAG}"   # e.g., hatespeech_LLM_gpt3_5


# =========================
# 4) Prompt (keep it SHORT for stability)
# =========================
SYSTEM_PROMPT = (
    "You are a content moderation classifier for hate speech.\n"
    "Output JSON only.\n"
)

# 你原来 10 条 criteria 很长，会增加输入 token，batch 大时更容易炸。
# 建议把判断规则压缩成一句话：任何“针对群体的贬损/侮辱/非人化/暴力/种族歧视”等 -> 1，否则 0。
RULE_HINT = (
  "Binary classification for child-safety moderation. "
  "Set label=1 ONLY when the text clearly contains hate/derogatory/abusive/harassing/bullying/dehumanizing/threatening/"
  "violent content toward a person or group (explicit or strongly implied), including slurs, clear targeted insults, "
  "dehumanization, calls for exclusion/discrimination, or encouragement/celebration of harm. "
  "Set label=0 if it is unclear/ambiguous, non-targeted profanity, general rudeness without a clear target, sarcasm without "
  "clear abuse, or mere discussion/quoting without endorsement. "
  "Tie-break: if unsure, choose 0. "
)


# =========================
# 5) Helpers
# =========================
def now_ts():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def ensure_dir(path: str):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

def build_user_prompt(texts):
    """
    Use JSON mode: MUST mention 'JSON' in the prompt.
    Return compact list to reduce output length.
    """
    n = len(texts)
    lines = [
        RULE_HINT,
        "",
        f"Return JSON only in EXACT format: {{\"predictions\":[0 or 1, ...]}}",
        f"Constraints: predictions length MUST be {n}, and MUST match the input order.",
        "",
        "Inputs:"
    ]
    for i, t in enumerate(texts):
        clean = str(t).replace("\n", " ")
        lines.append(f"{i}: {clean}")
    return "\n".join(lines)

def save_fail_log(fail_dir, texts, raw_output, err_msg):
    if not SAVE_FAIL_LOG:
        return
    ensure_dir(fail_dir)
    payload = {
        "error": err_msg,
        "n_texts": len(texts),
        "texts": texts,
        "raw_output": raw_output
    }
    fp = os.path.join(fail_dir, f"fail_{now_ts()}_n{len(texts)}.json")
    with open(fp, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

def is_rate_limit_error(e: Exception) -> bool:
    s = str(e).lower()
    return ("429" in s) or ("rate limit" in s) or ("quota" in s)

def call_model_json_object(texts, fail_dir=None):
    """
    Single API call for a list of texts.
    Must return a list of ints of SAME length as texts, or raise.
    """
    n = len(texts)
    user_prompt = build_user_prompt(texts)

    try:
        resp = client.responses.create(
            model=MODEL_NAME,
            input=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            # ✅ JSON mode (works for gpt-3.5 and also works for newer models)
            text={"format": {"type": "json_object"}},
            temperature=0.0,
            max_output_tokens=MAX_OUTPUT_TOKENS,
        )

        # Detect incomplete (important when output truncated)
        if getattr(resp, "status", None) == "incomplete":
            raise RuntimeError(f"incomplete output: {getattr(resp, 'incomplete_details', None)}")

        raw = resp.output_text
        data = json.loads(raw)
        preds = data.get("predictions", None)

        # Strict validation: length must match
        if (not isinstance(preds, list)) or len(preds) != n:
            raise ValueError(f"Bad predictions length: got {len(preds) if isinstance(preds, list) else type(preds)} expected {n}")

        # Validate values
        out = []
        for p in preds:
            if p not in (0, 1):
                raise ValueError(f"Bad prediction value: {p}")
            out.append(int(p))

        return out

    except Exception as e:
        # optional fail log
        raw_out = ""
        try:
            raw_out = resp.output_text if "resp" in locals() else ""
        except:
            raw_out = ""
        if fail_dir is not None:
            save_fail_log(fail_dir, texts, raw_out, str(e))
        raise

def classify_texts_robust(texts, fail_dir=None, max_same_batch_retries=3):
    """
    Robust classifier:
    - Try same batch a few times (handles transient noise/rate limit)
    - If still fails, split into halves recursively
    """
    if not texts:
        return []

    # 1) retry same size
    last_err = None
    for attempt in range(max_same_batch_retries):
        try:
            preds = call_model_json_object(texts, fail_dir=fail_dir)
            return preds
        except Exception as e:
            last_err = e
            # Rate limit -> wait longer
            if is_rate_limit_error(e):
                wait = 2 + attempt * 2
                print(f"\n⚠️ Rate limit. sleep {wait}s ...")
                time.sleep(wait)
            else:
                # Non rate-limit: small wait then retry
                time.sleep(0.5)

    # 2) if still fails -> split
    if len(texts) == 1:
        # 单条都失败：给 0（或者你也可以 raise）
        print(f"\n❌ Single item failed after retries. Defaulting to 0. Error: {last_err}")
        return [0]

    mid = len(texts) // 2
    left = classify_texts_robust(texts[:mid], fail_dir=fail_dir, max_same_batch_retries=max_same_batch_retries)
    right = classify_texts_robust(texts[mid:], fail_dir=fail_dir, max_same_batch_retries=max_same_batch_retries)
    return left + right


# =========================
# 6) File processing
# =========================
def process_single_file(file_path, output_dir, fail_dir):
    filename = os.path.basename(file_path)
    print(f"\nProcessing file: {filename}")

    stem, _ = os.path.splitext(filename)
    output_filename = f"{stem}{OUTPUT_SUFFIX}.csv"
    output_full_path = os.path.join(output_dir, output_filename)

    if os.path.exists(output_full_path):
        print(f"Skipping {filename} (Output already exists).")
        return

    try:
        df = pd.read_csv(file_path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding="latin1")

    if "text" not in df.columns:
        print(f"Skipping {filename} (No 'text' column found).")
        return

    results = []
    total = len(df)

    # 逐块取 BATCH_SIZE，但每块内部如果失败会自动拆分
    for start in tqdm(range(0, total, BATCH_SIZE), desc=f"Rows in {filename}", leave=False):
        batch = df.iloc[start : start + BATCH_SIZE]
        texts = batch["text"].tolist()

        preds = classify_texts_robust(texts, fail_dir=fail_dir, max_same_batch_retries=3)

        # 强保证：长度必须等于 texts
        if len(preds) != len(texts):
            raise RuntimeError(f"INTERNAL ERROR: preds len {len(preds)} != texts len {len(texts)}")

        results.extend(preds)
        time.sleep(SLEEP_BETWEEN_CALLS)

    df[OUTPUT_COL] = results
    df.to_csv(output_full_path, index=False, encoding="utf-8-sig")
    print(f"✅ Saved: {output_filename}")


def main():
    base_dir = os.path.dirname(INPUT_DIR)  # .../hs_sample_400
    output_dir = os.path.join(base_dir, OUTPUT_DIR_NAME)
    fail_dir = os.path.join(output_dir, "_fail_logs")

    ensure_dir(output_dir)
    ensure_dir(fail_dir)

    search_pattern = os.path.join(INPUT_DIR, "traindataset_p_*.csv")
    all_files = glob.glob(search_pattern)
    all_files.sort()

    print(f"Model: {MODEL_NAME}")
    print(f"Initial batch size: {BATCH_SIZE} (auto-split on failure)")
    print(f"Max output tokens: {MAX_OUTPUT_TOKENS}")
    print(f"Output dir: {output_dir}")
    print(f"Output column: {OUTPUT_COL}")
    print(f"Found {len(all_files)} files to process.")

    for file_path in tqdm(all_files, desc="Overall File Progress"):
        process_single_file(file_path, output_dir, fail_dir)

    print("\n" + "=" * 30)
    print("ALL FILES PROCESSED SUCCESSFULLY")
    print("=" * 30)


if __name__ == "__main__":
    main()


Model: gpt-5.2
Initial batch size: 50 (auto-split on failure)
Max output tokens: 2000
Output dir: D:\PythonEnv\Interpretable\hs_sample_400\GPTres_gpt_5_2
Output column: hatespeech_LLM_gpt_5_2
Found 11 files to process.


Overall File Progress:   0%|          | 0/11 [00:00<?, ?it/s]


Processing file: traindataset_p_00_sampled400_EXTREME_MIX_NOISE.csv


Rows in traindataset_p_00_sampled400_EXTREME_MIX_NOISE.csv:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Saved: traindataset_p_00_sampled400_EXTREME_MIX_NOISE_gpt_5_2.csv

Processing file: traindataset_p_01_sampled400_EXTREME_MIX_NOISE.csv


...

✅ Saved: traindataset_p_08_sampled400_EXTREME_MIX_NOISE_gpt_5_2.csv

Processing file: traindataset_p_09_sampled400_EXTREME_MIX_NOISE.csv


Rows in traindataset_p_09_sampled400_EXTREME_MIX_NOISE.csv:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Saved: traindataset_p_09_sampled400_EXTREME_MIX_NOISE_gpt_5_2.csv

Processing file: traindataset_p_10_sampled400_EXTREME_MIX_NOISE.csv


Rows in traindataset_p_10_sampled400_EXTREME_MIX_NOISE.csv:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Saved: traindataset_p_10_sampled400_EXTREME_MIX_NOISE_gpt_5_2.csv

ALL FILES PROCESSED SUCCESSFULLY


In [40]:
import os
import re
import random
import string
import numpy as np
import pandas as pd

# =========================
# CONFIG (你只需要改这里)
# =========================
INPUT_DIR = r"D:\PythonEnv\Interpretable\hs_sample_400\GPTResult"
BASE_FILE = "traindataset_p_00_sampled400_EXTREME_ZERO_TOL.csv"

# 输出目录：在 INPUT_DIR 同级新建一个文件夹
OUTPUT_FOLDER_NAME = "GPTResult_MIXED_NOISE"   # new folder
SEED = 42  # 固定随机种子保证可复现

# 噪音强度参数
CHAR_CHANGE_RATE = 0.5     # 在被扰动的文本中，字符替换比例（0.08=8%字符）
INWORD_SWAP_RATE = 0.50     # 在被扰动文本中，每个词发生swap的概率
WORD_DROP_RATE   = 0.08     # 词删除概率
WORD_DUP_RATE    = 0.08     # 词重复概率
WORD_SHUFFLE_PROB = 0.40    # 词序打乱概率（句子级）

# 输出文件名标签（会替换原文件名末尾的 ZERO_TOL）
OUTPUT_TAG = "EXTREME_MIX_NOISE"

# =========================
# Noise functions
# =========================
def _safe_text(x):
    if pd.isna(x):
        return ""
    return str(x)

def disturb_in_word(word: str, rng: np.random.Generator) -> str:
    """Swap adjacent chars inside a word or do a small shuffle."""
    if len(word) <= 3:
        return word
    # 50% swap, 50% small shuffle
    if rng.random() < 0.5:
        i = rng.integers(1, len(word) - 1)
        chars = list(word)
        chars[i], chars[i-1] = chars[i-1], chars[i]
        return "".join(chars)
    else:
        # keep first/last, shuffle middle
        mid = list(word[1:-1])
        rng.shuffle(mid)
        return word[0] + "".join(mid) + word[-1]

def char_change(text: str, rng: np.random.Generator, rate: float) -> str:
    """Randomly replace some characters with letters/digits/punct."""
    if not text:
        return text
    chars = list(text)
    n = len(chars)
    k = int(np.ceil(n * rate))
    if k <= 0:
        return text

    idxs = rng.choice(n, size=min(k, n), replace=False)

    alphabet = string.ascii_letters + string.digits
    for i in idxs:
        c = chars[i]
        # keep whitespace
        if c.isspace():
            continue
        # keep very common punctuation sometimes
        if c in ",.!?;:'\"()[]{}" and rng.random() < 0.7:
            continue
        # replace
        chars[i] = rng.choice(list(alphabet))
    return "".join(chars)

def word_disturb(text: str, rng: np.random.Generator) -> str:
    """Drop/duplicate/shuffle words."""
    if not text.strip():
        return text

    # split keeping simple whitespace tokens
    words = text.split()
    if len(words) <= 1:
        return text

    new_words = []
    for w in words:
        # drop
        if rng.random() < WORD_DROP_RATE:
            continue
        new_words.append(w)
        # duplicate
        if rng.random() < WORD_DUP_RATE:
            new_words.append(w)

    if len(new_words) <= 1:
        new_words = words[:]  # fallback

    # shuffle words sometimes
    if rng.random() < WORD_SHUFFLE_PROB and len(new_words) >= 3:
        rng.shuffle(new_words)

    return " ".join(new_words)

def mixed_noise_transform(text: str, rng: np.random.Generator) -> str:
    """
    Combine: in-word disturb + char-change + word disturb
    """
    text = _safe_text(text)
    if not text:
        return text

    # 1) word-level disturb
    text2 = word_disturb(text, rng)

    # 2) in-word disturb (apply per word with probability)
    words = text2.split()
    out_words = []
    for w in words:
        if rng.random() < INWORD_SWAP_RATE:
            out_words.append(disturb_in_word(w, rng))
        else:
            out_words.append(w)
    text3 = " ".join(out_words)

    # 3) char-change
    text4 = char_change(text3, rng, CHAR_CHANGE_RATE)
    return text4

# =========================
# Main generation
# =========================
def main():
    random.seed(SEED)
    np_rng = np.random.default_rng(SEED)

    input_path = os.path.join(INPUT_DIR, BASE_FILE)
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Base file not found: {input_path}")

    # output dir
    base_parent = os.path.dirname(INPUT_DIR)
    output_dir = os.path.join(base_parent, OUTPUT_FOLDER_NAME)
    os.makedirs(output_dir, exist_ok=True)

    # read base
    try:
        df0 = pd.read_csv(input_path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df0 = pd.read_csv(input_path, encoding="latin1")

    if "text" not in df0.columns:
        raise ValueError("Base CSV must contain a 'text' column.")

    n = len(df0)
    print(f"Loaded base file: {BASE_FILE}, rows={n}")
    print(f"Output folder: {output_dir}")

    # generate p_00 ~ p_10
    for k in range(0, 11):
        p = k / 10.0

        df = df0.copy(deep=True)

        # choose rows to perturb
        m = int(round(n * p))
        if m > 0:
            idxs = np_rng.choice(n, size=m, replace=False)
        else:
            idxs = np.array([], dtype=int)

        # apply transformation
        if m > 0:
            # Use a per-file rng for reproducibility across p-levels
            file_rng = np.random.default_rng(SEED + k * 1000)

            # transform selected rows
            for idx in idxs:
                df.at[idx, "text"] = mixed_noise_transform(df.at[idx, "text"], file_rng)

        # build output filename:
        # replace p_00 -> p_XX, and replace tail tag to OUTPUT_TAG
        out_name = BASE_FILE

        # p_00 -> p_XX
        out_name = re.sub(r"p_00", f"p_{k:02d}", out_name)

        # replace last tag "EXTREME_ZERO_TOL" -> OUTPUT_TAG (if exists)
        out_name = out_name.replace("EXTREME_ZERO_TOL", OUTPUT_TAG)

        out_path = os.path.join(output_dir, out_name)
        df.to_csv(out_path, index=False, encoding="utf-8-sig")

        print(f"✅ wrote {out_name} | p'={p:.1f} | perturbed_rows={m}/{n}")

    print("\nDONE. 11 files generated.")


if __name__ == "__main__":
    main()


Loaded base file: traindataset_p_00_sampled400_EXTREME_ZERO_TOL.csv, rows=400
Output folder: D:\PythonEnv\Interpretable\hs_sample_400\GPTResult_MIXED_NOISE
✅ wrote traindataset_p_00_sampled400_EXTREME_MIX_NOISE.csv | p'=0.0 | perturbed_rows=0/400
✅ wrote traindataset_p_01_sampled400_EXTREME_MIX_NOISE.csv | p'=0.1 | perturbed_rows=40/400
✅ wrote traindataset_p_02_sampled400_EXTREME_MIX_NOISE.csv | p'=0.2 | perturbed_rows=80/400
✅ wrote traindataset_p_03_sampled400_EXTREME_MIX_NOISE.csv | p'=0.3 | perturbed_rows=120/400
✅ wrote traindataset_p_04_sampled400_EXTREME_MIX_NOISE.csv | p'=0.4 | perturbed_rows=160/400
✅ wrote traindataset_p_05_sampled400_EXTREME_MIX_NOISE.csv | p'=0.5 | perturbed_rows=200/400
✅ wrote traindataset_p_06_sampled400_EXTREME_MIX_NOISE.csv | p'=0.6 | perturbed_rows=240/400
✅ wrote traindataset_p_07_sampled400_EXTREME_MIX_NOISE.csv | p'=0.7 | perturbed_rows=280/400
✅ wrote traindataset_p_08_sampled400_EXTREME_MIX_NOISE.csv | p'=0.8 | perturbed_rows=320/400
✅ wrote tra

In [2]:
import pandas as pd
import os
import glob
import re
import numpy as np   # ✅ add

# --- CONFIGURATION ---
BASE_DIR = r"D:\PythonEnv\Interpretable\hs_sample_400"

# ✅ Change this anytime, everything else follows automatically
MODEL_NAME = "gpt-5.2" #gpt-3.5-turbo

def model_to_tag(model_name: str) -> str:
    tag = model_name.lower()
    tag = re.sub(r"[^a-z0-9]+", "_", tag)
    tag = re.sub(r"_+", "_", tag).strip("_")
    return tag

MODEL_TAG = model_to_tag(MODEL_NAME)

RESULT_DIR = os.path.join(BASE_DIR, f"GPTres_{MODEL_TAG}")

FILE_SUFFIX = f"_{MODEL_TAG}.csv"
PRED_COL    = f"hatespeech_LLM_{MODEL_TAG}"

SUMMARY_OUT = f"summary_metrics_{MODEL_TAG}.csv"
OUT_P11_COL = f"{MODEL_TAG}_P11"
OUT_P00_COL = f"{MODEL_TAG}_P00"
OUT_PC_COL  = f"{MODEL_TAG}_Pc"

# ✅ add MI column
OUT_MI_COL  = f"{MODEL_TAG}_IYA"   # I(Y;A) in nats (ln base)

def _entropy_probs(probs):
    """Entropy H = -sum p log p, using natural log; treats 0*log(0)=0."""
    probs = np.asarray(probs, dtype=float)
    probs = probs[(probs > 0) & np.isfinite(probs)]
    if probs.size == 0:
        return 0.0
    return float(-(probs * np.log(probs)).sum())

def mutual_info_from_p11_p00(p1_truth, p11, p00):
    """
    Mutual information I(Y;A) (nats) for binary Y in {0,1} and A in {0,1},
    given p1 = P(Y=1), p11 = P(A=1|Y=1), p00 = P(A=0|Y=0).
    """
    if p11 is None or p00 is None:
        return None

    p1 = float(p1_truth)
    p0 = 1.0 - p1

    # Conditional confusion entries
    p10 = 1.0 - float(p11)  # P(A=0|Y=1)
    p01 = 1.0 - float(p00)  # P(A=1|Y=0)

    # Marginal P(A=1), P(A=0)
    pA1 = p1 * float(p11) + p0 * p01
    pA0 = 1.0 - pA1

    H_A = _entropy_probs([pA0, pA1])
    H_A_given_Y1 = _entropy_probs([p10, float(p11)])
    H_A_given_Y0 = _entropy_probs([float(p00), p01])

    H_A_given_Y = p1 * H_A_given_Y1 + p0 * H_A_given_Y0

    I_YA = H_A - H_A_given_Y
    return float(I_YA)

def calculate_metrics_for_model(df, truth_col, pred_col, p1_truth, p0_truth):
    """Calculates Sensitivity (P11), Specificity (P00), and Accuracy (Pc)."""
    if pred_col not in df.columns:
        return None, None, None

    pred = pd.to_numeric(df[pred_col], errors="coerce").fillna(0).astype(int)
    pred = pred.clip(lower=0, upper=1)

    truth_1 = df[df[truth_col] == 1]
    p11 = pred.loc[truth_1.index].mean() if len(truth_1) > 0 else 0.0

    truth_0 = df[df[truth_col] == 0]
    p00 = (1.0 - pred.loc[truth_0.index].mean()) if len(truth_0) > 0 else 0.0

    pc = (p1_truth * p11) + (p0_truth * p00)
    return float(p11), float(p00), float(pc)

def get_p_value_from_filename(filename):
    """Extracts p from 'traindataset_p_05_...' -> 0.5"""
    match = re.search(r"_p_(\d+)_", filename)
    if match:
        num = int(match.group(1))
        return num / 10.0
    return -1

def main():
    if not os.path.exists(RESULT_DIR):
        print(f"Error: Directory not found: {RESULT_DIR}")
        return

    search_pattern = os.path.join(RESULT_DIR, f"*{FILE_SUFFIX}")
    files = glob.glob(search_pattern)

    if not files:
        print(f"No result files found for model={MODEL_NAME}")
        print(f"Expected pattern: {search_pattern}")
        print(f"Expected prediction column: {PRED_COL}")
        return

    summary_data = []
    print(f"Model: {MODEL_NAME}")
    print(f"Result dir: {RESULT_DIR}")
    print(f"File pattern: {search_pattern}")
    print(f"Prediction column: {PRED_COL}")
    print(f"Found {len(files)} files. Analyzing...")

    for file_path in files:
        filename = os.path.basename(file_path)

        try:
            df = pd.read_csv(file_path, encoding="utf-8-sig")
        except:
            df = pd.read_csv(file_path, encoding="latin1")

        p_val = get_p_value_from_filename(filename)

        if "hs_state" not in df.columns:
            print(f"Skipping {filename}: 'hs_state' missing.")
            continue

        p1_truth = float(df["hs_state"].mean())
        p0_truth = 1.0 - p1_truth

        p11, p00, pc = calculate_metrics_for_model(
            df, "hs_state", PRED_COL, p1_truth, p0_truth
        )

        # ✅ Mutual information I(Y;A) (nats)
        iya = mutual_info_from_p11_p00(p1_truth, p11, p00)

        summary_data.append({
            "File_P": p_val,
            "Filename": filename,
            OUT_P11_COL: p11,
            OUT_P00_COL: p00,
            OUT_PC_COL:  pc,
            OUT_MI_COL:  iya,   # ✅ add
        })

    summary_df = pd.DataFrame(summary_data).sort_values(by="File_P")

    display_cols = ["File_P", OUT_P11_COL, OUT_P00_COL, OUT_PC_COL, OUT_MI_COL]
    print("\n" + "="*80)
    print(f"{(' ' + MODEL_NAME + ' METRICS '):=^80}")
    print("="*80)
    print(summary_df[display_cols].round(6).to_string(index=False))

    output_path = os.path.join(RESULT_DIR, SUMMARY_OUT)
    summary_df.to_csv(output_path, index=False)
    print("\n" + "="*80)
    print(f"Summary saved to: {output_path}")

if __name__ == "__main__":
    main()


Model: gpt-5.2
Result dir: D:\PythonEnv\Interpretable\hs_sample_400\GPTres_gpt_5_2
File pattern: D:\PythonEnv\Interpretable\hs_sample_400\GPTres_gpt_5_2\*_gpt_5_2.csv
Prediction column: hatespeech_LLM_gpt_5_2
Found 12 files. Analyzing...
Skipping summary_metrics_gpt_5_2.csv: 'hs_state' missing.

=============================== gpt-5.2 METRICS ================================
 File_P  gpt_5_2_P11  gpt_5_2_P00  gpt_5_2_Pc  gpt_5_2_IYA
    0.0        0.855        0.880      0.8675     0.302404
    0.1        0.820        0.870      0.8450     0.263007
    0.2        0.690        0.875      0.7825     0.178000
    0.3        0.665        0.895      0.7800     0.179661
    0.4        0.555        0.920      0.7375     0.142044
    0.5        0.505        0.935      0.7200     0.130811
    0.6        0.415        0.965      0.6900     0.117993
    0.7        0.320        0.965      0.6425     0.078289
    0.8        0.250        0.960      0.6050     0.048797
    0.9        0.175        0.96